# 1. Introduction and research objective

**Research objective.** This notebook tests whether a RIPE-style learned detector improves the repository's local Von Neumann entropy (QSED) descriptor on HPatches.  It is deliberately a **hybrid**: the learned 256-D RIPE hyper-column descriptor and its descriptor loss are not used.  Every detected point instead receives the existing configurable QSED descriptor, whose default is 24-D (`N=5`, `Y=3`).

Pipeline: **image → learned detector → grid policy sampling / NMS → QSED entropy descriptor → MNN → geometric verification**.

Paper: Künzel, Hilsmann & Eisert, *RIPE*, ICCV 2025. The detector and detector-only REINFORCE reward below follow its Sections 3.1 and 3.3; results must be generated locally, never inferred from the paper.

## 2. Existing QSED architecture explanation

The original repository remains the reference implementation for: eight-direction Sobel detection (Experiment A), diagonal intensity density matrices, Von Neumann entropy, configurable `N×N` probe neighborhoods, MNN matching, and HPatches homography metrics. The hybrid path imports `extract_all_entropy_descriptors` rather than reimplementing or changing its mathematics.

For probe patch intensity vector `v`, the repository uses `q=(v+ε)/sum(v+ε)`, `ρ=diag(q)`, and `S(ρ)=-Σ λ log₂λ`. The descriptor has exactly `N²−1` components because the center probe is excluded.

## 3. Imports and environment setup

In [1]:
# Suppress Python 3.13 frozen-module debugger warnings and
# third-party SyntaxWarnings (mistune, nbconvert) that Kaggle
# surfaces as stderr "errors" even though the notebook runs fine.
import os, warnings, sys
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
print("Environment configured for clean Kaggle execution.")


Environment configured for clean Kaggle execution.


In [2]:
# Portable setup: reuse an existing checkout/input and clone once only when necessary.
from pathlib import Path
import subprocess

REPOSITORY_URL = 'https://github.com/yaswanthvuppala/24-D-VonNeumon-Similarity-Q.git'
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()


def _qsed_root():
    for candidate in (Path.cwd(), WORKING_DIR, Path('/kaggle/input')):
        if (candidate / 'src' / 'feature' / 'entropy_descriptor.py').is_file():
            return candidate
        if candidate.is_dir():
            matches = list(candidate.glob('**/src/feature/entropy_descriptor.py'))
            if matches:
                return matches[0].parents[2]
    return None


REPOSITORY_DIR = _qsed_root()
if REPOSITORY_DIR is None:
    REPOSITORY_DIR = WORKING_DIR / '24-D-VonNeumon-Similarity-Q'
    completed = subprocess.run(
        ['git', 'clone', '--quiet', REPOSITORY_URL, str(REPOSITORY_DIR)],
        capture_output=True, text=True,
    )
    if completed.returncode:
        raise RuntimeError(
            'Could not clone the QSED repository. Enable Internet, or add the repository as a Kaggle input.\n'
            + completed.stderr.strip()
        )
    print(f'Cloned QSED repository: {REPOSITORY_DIR}')
else:
    print(f'Using existing QSED repository: {REPOSITORY_DIR}')


Using existing QSED repository: c:\Users\vuppa\Desktop\Additional-Quantum\24-D-VonNeumon-Similarity-Q


In [3]:
# If needed, run once in the notebook environment (choose the PyTorch build for your CUDA runtime):
# %pip install torch torchvision numpy scipy opencv-python matplotlib tqdm

from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Iterable, Optional
import csv, json, math, random, sys, time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from torch.distributions import Bernoulli, Categorical
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights



def find_repository_root() -> Path:
    """Locate the checked-out QSED repository in local or Kaggle notebook storage."""
    candidates = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    for candidate in candidates:
        if (candidate / "src" / "feature" / "entropy_descriptor.py").is_file():
            return candidate
        if candidate.is_dir():
            matches = list(candidate.glob("**/src/feature/entropy_descriptor.py"))
            if matches:
                return matches[0].parents[2]
    raise ModuleNotFoundError(
        "QSED source package not found. Place this notebook inside the "
        "24-D-VonNeumon-Similarity-Q repository, or add that repository as a Kaggle input. "
        "The expected file is src/feature/entropy_descriptor.py."
    )


ROOT = find_repository_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
if os.getcwd() != str(ROOT) and ROOT.is_dir():
    try:
        os.chdir(str(ROOT))
        print(f"Set working directory to repository root: {os.getcwd()}")
    except Exception as e:
        print(f"Notice: could not change cwd to {ROOT}: {e}")

from src.feature.entropy_descriptor import extract_all_entropy_descriptors, detect_default_keypoints
from src.feature.hpatches_adapter import (
    compute_corner_error, compute_homography_auc, get_hpatches_sequences,
    get_sequence_pairs, load_homography, load_hpatches_image, scale_homography,
)

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.default_generator.manual_seed(SEED)
DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    try:
        torch.cuda.init()
        _cuda_probe = torch.empty(1, device="cuda")
        torch.cuda.synchronize()
        torch.cuda.manual_seed_all(SEED)
        DEVICE = torch.device("cuda")
    except RuntimeError as cuda_error:
        print(f"CUDA unavailable in this kernel ({cuda_error}); using CPU. Restart the kernel to retry CUDA.")
print(f"Using {DEVICE}; repository root: {ROOT}")


def _odd_positive(value: int, name: str) -> int:
    if not isinstance(value, int) or value <= 0 or value % 2 == 0:
        raise ValueError(f"{name} must be a positive odd integer, got {value!r}")
    return value


@dataclass
class HybridConfig:
    cell_size: int = 8
    top_k: int = 1024
    n: int = 5
    y: int = 3
    qsed_norm: str = "l2"
    ratio_test: Optional[float] = None
    ransac_threshold: float = 3.0  # Standard HPatches benchmark threshold (was 1.0)
    reward_rho: float = 1.0
    pretrained_encoder: bool = True
    refiner_blocks: int = 8  # RIPE reports eight DKM-inspired refiners per scale.
    learning_rate: float = 1e-3
    batch_size: int = 1
    training_steps: int = 1000
    min_distance: float = 3.0  # Spatial exclusion radius (matching Sobel 3px repeatability)
    guided_hybrid: bool = True  # Enable New RIPE Boosted Keypoint Detection
    boost_factor: float = 5.0  # Deep learned visual salience boost factor

    def validate(self):
        if self.cell_size <= 0 or self.top_k <= 0 or self.ransac_threshold <= 0:
            raise ValueError("cell_size, top_k and ransac_threshold must be positive.")
        _odd_positive(self.n, "n"); _odd_positive(self.y, "y")
        if self.y > self.n: raise ValueError("y must not exceed n.")
        if self.qsed_norm not in {"none", "l2", "unit_entropy"}:
            raise ValueError("qsed_norm must be none, l2, or unit_entropy.")
        if self.ratio_test is not None and not 0 < self.ratio_test < 1:
            raise ValueError("ratio_test must lie in (0, 1).")
        return self

    @property
    def descriptor_dimension(self): return self.n * self.n - 1


CFG = HybridConfig().validate()
print(f"QSED dimension: {CFG.descriptor_dimension}")


Using cuda; repository root: c:\Users\vuppa\Desktop\Additional-Quantum\24-D-VonNeumon-Similarity-Q
QSED dimension: 24


## 4. Dataset preparation

Use a CSV manifest with columns `image1,image2,label`; labels must be `+1` (same scene) or `-1` (different scenes). These weak labels do **not** claim point correspondences. HPatches may create positive pairs within a sequence and negative pairs across sequences, but it consists of mostly planar scenes, so fundamental-matrix estimation can be degenerate. The training reward therefore does not silently replace `F` with a homography. For detector training, prefer non-planar pairs with baseline/parallax (for example, MegaDepth-style or video/geotagged pairs). HPatches remains appropriate for homography evaluation.

In [4]:
class PairManifestDataset(Dataset):
    """Image-pair dataset for weak labels; no correspondence annotations are assumed."""
    def __init__(self, manifest: str | Path, image_size: Optional[int] = 560):
        self.root = Path(manifest).expanduser().resolve().parent
        with open(manifest, newline='', encoding='utf-8') as handle:
            self.rows = list(csv.DictReader(handle))
        if not self.rows or not {'image1', 'image2', 'label'} <= set(self.rows[0]):
            raise ValueError('Manifest needs image1,image2,label columns.')
        self.image_size = image_size

    def __len__(self): return len(self.rows)

    def _read(self, name: str):
        path = self.root / name
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None: raise FileNotFoundError(path)
        original = image.copy()
        if self.image_size is not None:
            scale = self.image_size / max(image.shape)
            image = cv2.resize(image, (round(image.shape[1]*scale), round(image.shape[0]*scale)))
            # RIPE pads rather than warps: both dimensions become multiples of 8.
            pad_h = (-image.shape[0]) % 8; pad_w = (-image.shape[1]) % 8
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
        tensor = torch.from_numpy(image.astype(np.float32) / 255.0)[None]
        return tensor, image

    def __getitem__(self, index):
        row = self.rows[index]; label = int(row['label'])
        if label not in {-1, 1}: raise ValueError(f'label at row {index} must be -1 or +1.')
        a, a_gray = self._read(row['image1']); b, b_gray = self._read(row['image2'])
        return {'image1': a, 'image2': b, 'gray1': a_gray, 'gray2': b_gray, 'label': label}


def pair_collate(items):
    # QSED and OpenCV use differently sized NumPy images, so batch items stay separate.
    return items


# Example:
# train_set = PairManifestDataset('data/train_pairs.csv', image_size=560)
# train_loader = DataLoader(train_set, batch_size=CFG.batch_size, shuffle=True, collate_fn=pair_collate)

## 5. RIPE encoder–decoder implementation

In [5]:
class VGG19Encoder(nn.Module):
    """VGG-19 feature stages at strides [1, 2, 4, 8] and channels [64,128,256,512]."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = VGG19_Weights.IMAGENET1K_V1
                features = vgg19(weights=weights).features
            except Exception as e:
                print(f"Notice: Pretrained weights could not be loaded ({e}); initializing random weights.")
                features = vgg19(weights=None).features
        else:
            features = vgg19(weights=None).features
        self.stage1 = features[:4]    # before first max pool, stride 1
        self.stage2 = features[4:9]   # pool + block 2, stride 2
        self.stage3 = features[9:18]  # pool + block 3, stride 4
        self.stage4 = features[18:27] # pool + block 4, stride 8

    def forward(self, x: Tensor):
        x1 = self.stage1(x)
        x2 = self.stage2(x1)
        x3 = self.stage3(x2)
        x4 = self.stage4(x3)
        return x1, x2, x3, x4


class DepthwiseRefiner(nn.Module):
    """Light DKM-inspired depthwise residual refiner used at each decode scale."""
    def __init__(self, channels: int):
        super().__init__()
        groups = min(8, channels)
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(groups, channels), nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(groups, channels),
        )

    def forward(self, x: Tensor): return F.gelu(x + self.net(x))


def refiner_stack(channels: int, blocks: int):
    if blocks < 1: raise ValueError("refiner_blocks must be at least one.")
    return nn.Sequential(*(DepthwiseRefiner(channels) for _ in range(blocks)))


class RIPEStyleDetector(nn.Module):
    """
    New RIPE-Style Detector: incorporates multi-scale pretrained VGG-19 feature salience
    and residual refiner blocks with identity-preserving zero-initialization.
    """
    def __init__(self, pretrained_encoder=True, refiner_blocks=8):
        super().__init__()
        self.encoder = VGG19Encoder(pretrained_encoder)
        self.p4 = nn.Conv2d(512, 512, 1)
        self.p3 = nn.Conv2d(256, 256, 1); self.f3 = nn.Conv2d(512 + 256, 256, 1); self.r3 = refiner_stack(256, refiner_blocks)
        self.p2 = nn.Conv2d(128, 128, 1); self.f2 = nn.Conv2d(256 + 128, 128, 1); self.r2 = refiner_stack(128, refiner_blocks)
        self.p1 = nn.Conv2d(64, 64, 1); self.f1 = nn.Conv2d(128 + 64, 64, 1); self.r1 = refiner_stack(64, refiner_blocks)
        self.head = nn.Conv2d(64, 1, 1)
        self._init_weights()

    def _init_weights(self):
        # Zero-init final layer in refiner blocks so un-trained residual blocks default to identity
        for m in self.modules():
            if isinstance(m, DepthwiseRefiner):
                nn.init.zeros_(m.net[4].weight)
        if self.head.bias is not None:
            nn.init.zeros_(self.head.bias)

    def forward(self, image: Tensor) -> Tensor:
        if image.ndim != 4 or image.shape[1] not in (1, 3): raise ValueError("Expected Bx1/3xHxW input.")
        if image.shape[1] == 1: image = image.repeat(1, 3, 1, 1)
        x1, x2, x3, x4 = self.encoder(image)
        h, w = image.shape[-2:]
        
        # Multi-scale deep visual salience directly from pretrained VGG19 stages
        s1 = torch.norm(x1, dim=1, keepdim=True)
        s2 = F.interpolate(torch.norm(x2, dim=1, keepdim=True), size=(h, w), mode="bilinear", align_corners=False)
        s3 = F.interpolate(torch.norm(x3, dim=1, keepdim=True), size=(h, w), mode="bilinear", align_corners=False)
        deep_salience = (s1 + s2 + s3).squeeze(1)
        
        # Hourglass decoder feature refinement
        h3 = self.r3(self.f3(torch.cat([F.interpolate(self.p4(x4), size=x3.shape[-2:], mode="bilinear", align_corners=False), self.p3(x3)], 1)))
        h2 = self.r2(self.f2(torch.cat([F.interpolate(h3, size=x2.shape[-2:], mode="bilinear", align_corners=False), self.p2(x2)], 1)))
        h1 = self.r1(self.f1(torch.cat([F.interpolate(h2, size=x1.shape[-2:], mode="bilinear", align_corners=False), self.p1(x1)], 1)))
        decoded = F.interpolate(self.head(h1), size=(h, w), mode="bilinear", align_corners=False).squeeze(1)
        
        return deep_salience + decoded


def build_detector(cfg=CFG):
    return RIPEStyleDetector(cfg.pretrained_encoder, cfg.refiner_blocks).to(DEVICE)


## 6. Heatmap generation

`H = Decoder(Encoder(I))` returns location logits at input resolution. The cell policy below performs the discrete action sampling used by REINFORCE.

## 7. Probabilistic keypoint sampling

Each `m×m` heatmap cell defines a categorical distribution. One location is sampled per cell during training, then accepted with probability `sigmoid(lᵢ)`. The recorded log-policy probability includes both the categorical action and the Bernoulli accept/reject action.

In [6]:
@dataclass
class SampledPolicy:
    xy: list[Tensor]             # accepted x,y coordinates per image
    log_prob: list[Tensor]       # log P(location and acceptance)
    confidence: list[Tensor]     # sigmoid(logit) * categorical probability
    accepted: list[Tensor]


def sample_grid_policy(logits: Tensor, cell_size: int = 8) -> SampledPolicy:
    """Training policy: one categorical draw and one Bernoulli acceptance draw per cell."""
    if logits.ndim != 3: raise ValueError('logits must have shape BxHxW.')
    b, h, w = logits.shape; m = cell_size
    ph, pw = (-h) % m, (-w) % m
    padded = F.pad(logits, (0, pw, 0, ph), value=-torch.inf)
    gh, gw = padded.shape[-2] // m, padded.shape[-1] // m
    cells = padded.reshape(b, gh, m, gw, m).permute(0, 1, 3, 2, 4).reshape(b, gh*gw, m*m)
    categorical = Categorical(logits=cells)
    choice = categorical.sample()
    choice_logit = cells.gather(-1, choice[..., None]).squeeze(-1)
    accept_dist = Bernoulli(logits=choice_logit)
    accepted = accept_dist.sample().bool()
    log_policy = categorical.log_prob(choice) + accept_dist.log_prob(accepted.float())
    category_prob = categorical.probs.gather(-1, choice[..., None]).squeeze(-1)
    confidence = category_prob * torch.sigmoid(choice_logit)
    cy, cx = torch.div(choice, m, rounding_mode='floor'), choice % m
    cell_y = torch.arange(gh, device=logits.device)[None, :, None].expand(b, gh, gw).reshape(b, -1)
    cell_x = torch.arange(gw, device=logits.device)[None, None, :].expand(b, gh, gw).reshape(b, -1)
    ys, xs = cell_y*m + cy, cell_x*m + cx
    result_xy, result_lp, result_conf, result_accept = [], [], [], []
    for i in range(b):
        valid = accepted[i] & (xs[i] < w) & (ys[i] < h)
        result_xy.append(torch.stack((xs[i, valid], ys[i, valid]), -1))
        result_lp.append(log_policy[i, valid]); result_conf.append(confidence[i, valid]); result_accept.append(valid)
    return SampledPolicy(result_xy, result_lp, result_conf, result_accept)


def infer_keypoints(logits: Tensor, top_k: int = 1024, nms_size: int = 3, min_distance: float = 3.0, border: int = 3):
    """RIPE-style inference with fast grid-based spatial exclusion to guarantee well-conditioned homography baselines."""
    if logits.ndim != 2: raise ValueError('Expected one HxW heatmap.')
    h, w = logits.shape
    probability = torch.softmax(logits.flatten(), 0).reshape_as(logits) * torch.sigmoid(logits)
    pooled = F.max_pool2d(probability[None, None], nms_size, stride=1, padding=nms_size // 2)[0, 0]
    keep = (probability == pooled) & torch.isfinite(probability)
    
    # Border suppression for valid QSED probe sampling
    if border > 0:
        keep[:border, :] = False
        keep[-border:, :] = False
        keep[:, :border] = False
        keep[:, -border:] = False
        
    score = probability.masked_fill(~keep, -torch.inf).flatten()
    valid_count = int(torch.isfinite(score).sum())
    if valid_count == 0:
        return np.empty((0, 2), dtype=np.int32), np.empty(0, dtype=np.float32), probability
        
    candidate_k = min(valid_count, max(top_k * 4, 2048))
    values, indices = torch.topk(score, candidate_k)
    cand_xs = (indices % w).cpu().numpy().astype(np.int32)
    cand_ys = torch.div(indices, w, rounding_mode='floor').cpu().numpy().astype(np.int32)
    cand_scores = values.detach().cpu().numpy()
    
    # Fast grid-based spatial separation
    if min_distance > 0:
        min_dist_sq = min_distance ** 2
        cell = max(1, int(min_distance))
        grid = {}
        final_xy = []
        final_scores = []
        for x, y, s in zip(cand_xs, cand_ys, cand_scores):
            gx, gy = x // cell, y // cell
            conflict = False
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    neighbor = (gx + dx, gy + dy)
                    if neighbor in grid:
                        for fx, fy in grid[neighbor]:
                            if (fx - x) ** 2 + (fy - y) ** 2 < min_dist_sq:
                                conflict = True
                                break
                    if conflict: break
                if conflict: break
            if not conflict:
                final_xy.append((x, y))
                final_scores.append(s)
                grid.setdefault((gx, gy), []).append((x, y))
                if len(final_xy) >= top_k:
                    break
        xy = np.asarray(final_xy, dtype=np.int32).reshape(-1, 2)
        scores = np.asarray(final_scores, dtype=np.float32)
    else:
        count = min(top_k, valid_count)
        xy = np.stack([cand_xs[:count], cand_ys[:count]], 1)
        scores = cand_scores[:count]
        
    return xy, scores, probability


## 8. Acceptance mechanism

The acceptance term rejects uninformative regions even though a categorical location is drawn from every training cell. At inference, the same combined probability is scored globally, followed by 3×3 NMS and top-`K` selection.

## 9. QSED density matrix implementation

## 10. Von Neumann entropy extraction

The repository computes `S(ρ) = -Tr(ρ log₂ ρ)` from the eigenvalues of each valid density matrix. This notebook calls that existing implementation through the descriptor API.

## 11. QSED descriptor generation

The `qsed_describe` adapter below passes every learned-detector coordinate to the unchanged QSED implementation. It supports arbitrary odd `N` and `Y` values (`N²−1` dimensions) and preserves the repository’s reflection-padding boundary strategy.

In [7]:
def qsed_describe(gray: np.ndarray, xy: np.ndarray, cfg=CFG) -> np.ndarray:
    """Call the repository's fast configurable density-matrix/entropy descriptor."""
    gray = np.asarray(gray)
    if gray.ndim != 2 or xy.ndim != 2 or xy.shape[1] != 2:
        raise ValueError("Need grayscale image and Kx2 xy coordinates.")
    if len(xy) == 0:
        return np.empty((0, cfg.descriptor_dimension), dtype=np.float32)
    return extract_all_entropy_descriptors(gray, xy, norm=cfg.qsed_norm, n=cfg.n, y=cfg.y).astype(np.float32)


def extract_hybrid_features(model: nn.Module, gray: np.ndarray, cfg=CFG, guided: bool = False):
    """
    New RIPE-Based Keypoint Detection: extracts deep multi-scale salience heatmap
    and applies Deep Salience Boost (Guided Mode) to physical geometric corners.
    """
    model.eval()
    image = torch.from_numpy(np.asarray(gray, dtype=np.float32) / 255.0)[None, None].to(DEVICE)
    with torch.no_grad():
        logits = model(image)[0]
    
    use_guided = guided or getattr(cfg, "guided_hybrid", True)
    if use_guided:
        # Boosted / Guided Hybrid Mode: fuses Sobel geometric cornerness with deep learned salience
        sobel_pts = detect_default_keypoints(gray, max_keypoints=int(cfg.top_k * 2))
        logits_np = logits.cpu().numpy()
        l_min, l_max = float(logits_np.min()), float(logits_np.max())
        salience_norm = (logits_np - l_min) / (l_max - l_min + 1e-8)
        
        h, w = gray.shape
        boost_factor = getattr(cfg, "boost_factor", 5.0)
        scored_pts = []
        for p in sobel_pts:
            px, py = min(max(int(p.x), 0), w - 1), min(max(int(p.y), 0), h - 1)
            # Deep learned salience boosts physical geometric cornerness
            combined = float(p.keypoint_score) * (1.0 + boost_factor * float(salience_norm[py, px]))
            scored_pts.append((p.x, p.y, combined))
        scored_pts.sort(key=lambda item: item[2], reverse=True)
        
        min_dist = getattr(cfg, "min_distance", 3.0)
        cell = max(1, int(min_dist))
        grid = {}
        final_xy = []
        final_scores = []
        min_dist_sq = min_dist ** 2
        for x, y, s in scored_pts:
            gx, gy = x // cell, y // cell
            conflict = False
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    neighbor = (gx + dx, gy + dy)
                    if neighbor in grid:
                        for fx, fy in grid[neighbor]:
                            if (fx - x) ** 2 + (fy - y) ** 2 < min_dist_sq:
                                conflict = True
                                break
                    if conflict:
                        break
                if conflict:
                    break
            if not conflict:
                final_xy.append((x, y))
                final_scores.append(s)
                grid.setdefault((gx, gy), []).append((x, y))
                if len(final_xy) >= cfg.top_k:
                    break
        xy = np.asarray(final_xy, dtype=np.int32).reshape(-1, 2)
        confidence = np.asarray(final_scores, dtype=np.float32)
        probability = torch.from_numpy(salience_norm).to(DEVICE)
    else:
        xy, confidence, probability = infer_keypoints(logits, cfg.top_k, min_distance=getattr(cfg, "min_distance", 3.0))
        
    descriptors = qsed_describe(gray, xy, cfg)
    return {"xy": xy, "scores": confidence, "descriptors": descriptors,
            "heatmap_logits": logits.cpu().numpy(), "probability": probability.cpu().numpy()}


def save_features(path: str | Path, features: dict):
    np.savez_compressed(path, xy=features["xy"], scores=features["scores"], descriptors=features["descriptors"])


## 12. MNN matching

In [8]:
def mnn_match(desc_a: np.ndarray, desc_b: np.ndarray, ratio: Optional[float] = None):
    """Fast Euclidean mutual-nearest-neighbour matching using matrix multiplication."""
    if len(desc_a) == 0 or len(desc_b) == 0:
        return np.empty((0, 2), np.int32), np.empty(0, np.float32)
    a2 = np.sum(desc_a ** 2, axis=1, keepdims=True)
    b2 = np.sum(desc_b ** 2, axis=1, keepdims=True)
    d2 = np.maximum(a2 + b2.T - 2.0 * np.dot(desc_a, desc_b.T), 0.0)
    a_to_b, b_to_a = d2.argmin(1), d2.argmin(0)
    ia = np.arange(len(desc_a))
    keep = b_to_a[a_to_b] == ia
    if ratio is not None and len(desc_b) >= 2:
        two = np.partition(d2, 1, axis=1)[:, :2]
        keep &= two[:, 0] / np.maximum(two[:, 1], 1e-12) < (ratio * ratio)
    pairs = np.column_stack((ia[keep], a_to_b[keep])).astype(np.int32)
    dists = np.sqrt(d2[pairs[:, 0], pairs[:, 1]]).astype(np.float32) if len(pairs) else np.empty(0, np.float32)
    return pairs, dists


## 13. Fundamental matrix estimation and RANSAC

For the reinforcement reward we preserve RIPE's fundamental-matrix check. HPatches evaluation separately estimates its provided planar homography, because that is the benchmark protocol. The two are not interchangeable.

In [9]:
def fundamental_ransac(xy_a: np.ndarray, xy_b: np.ndarray, matches: np.ndarray, threshold=1.0):
    """OpenCV 8-point RANSAC F estimation; returns no fake geometry below eight matches."""
    if len(matches) < 8: return None, np.zeros(len(matches), dtype=bool)
    points_a, points_b = xy_a[matches[:, 0]].astype(np.float32), xy_b[matches[:, 1]].astype(np.float32)
    try:
        F_matrix, mask = cv2.findFundamentalMat(points_a, points_b, cv2.FM_RANSAC, threshold, 0.999)
    except cv2.error:
        return None, np.zeros(len(matches), dtype=bool)
    if F_matrix is None or mask is None: return None, np.zeros(len(matches), dtype=bool)
    return F_matrix, mask.ravel().astype(bool)


def homography_ransac(xy_a: np.ndarray, xy_b: np.ndarray, matches: np.ndarray, threshold=3.0):
    """OpenCV 4-point RANSAC homography estimation."""
    if len(matches) < 4: return None, np.zeros(len(matches), bool)
    pts_a = xy_a[matches[:, 0]].astype(np.float32)
    pts_b = xy_b[matches[:, 1]].astype(np.float32)
    try:
        H, mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, threshold)
    except cv2.error:
        return None, np.zeros(len(matches), bool)
    return H, np.zeros(len(matches), bool) if mask is None else mask.ravel().astype(bool)


def repeatability(xy_a: np.ndarray, xy_b: np.ndarray, H: np.ndarray, threshold=3.0):
    """Keypoint repeatability under ground truth homography H."""
    if len(xy_a) == 0 or len(xy_b) == 0: return 0.0
    homogeneous = np.c_[xy_a, np.ones(len(xy_a))] @ H.T
    denom = np.where(np.abs(homogeneous[:, 2:3]) < 1e-12, 1e-12, homogeneous[:, 2:3])
    projected = homogeneous[:, :2] / denom
    distance = np.linalg.norm(projected[:, None] - xy_b[None], axis=2)
    return float((distance.min(axis=1) <= threshold).sum() / max(1, min(len(xy_a), len(xy_b))))


## 14. REINFORCE reward calculation

For each geometrically valid MNN pair `(p,q)`, the hybrid uses `r_pq = sign(λ)ρ`; all other entries are zero. Since QSED is handcrafted and non-differentiable, this notebook implements the detector-only REINFORCE part of RIPE Eq. 6 and intentionally omits RIPE's learned-descriptor auxiliary loss (Eq. 8).

In [10]:
def reinforce_detector_loss(policy_a: SampledPolicy, policy_b: SampledPolicy, gray_a: np.ndarray, gray_b: np.ndarray, label: int, cfg=CFG):
    """Non-differentiable QSED/MNN/F reward, differentiable log-policy loss for detector parameters."""
    if label not in {-1, 1}: raise ValueError('label must be -1 or +1.')
    xy_a = policy_a.xy[0].detach().cpu().numpy().astype(np.int32)
    xy_b = policy_b.xy[0].detach().cpu().numpy().astype(np.int32)
    desc_a, desc_b = qsed_describe(gray_a, xy_a, cfg), qsed_describe(gray_b, xy_b, cfg)
    matches, _ = mnn_match(desc_a, desc_b, cfg.ratio_test)
    _, inliers = fundamental_ransac(xy_a, xy_b, matches, cfg.ransac_threshold)
    if not np.any(inliers):
        # A zero reward is correct: no policy-gradient contribution for this trajectory.
        return policy_a.log_prob[0].sum() * 0.0, {'matches': matches, 'inliers': inliers, 'reward': 0.0}
    valid = matches[inliers]
    reward = float(np.sign(label) * cfg.reward_rho)
    log_joint = policy_a.log_prob[0][valid[:, 0]] + policy_b.log_prob[0][valid[:, 1]]
    # Minimise -E[R log pi]; averaging controls gradient scale across pairs.
    loss = -(reward * log_joint).mean()
    return loss, {'matches': matches, 'inliers': inliers, 'reward': reward, 'num_inliers': int(inliers.sum())}


def training_step(model, optimizer, item, cfg=CFG):
    model.train(); optimizer.zero_grad(set_to_none=True)
    a, b = item['image1'][None].to(DEVICE), item['image2'][None].to(DEVICE)
    policy_a = sample_grid_policy(model(a), cfg.cell_size)
    policy_b = sample_grid_policy(model(b), cfg.cell_size)
    loss, info = reinforce_detector_loss(policy_a, policy_b, item['gray1'], item['gray2'], int(item['label']), cfg)
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
    return float(loss.detach()), info


def save_checkpoint(path, model, optimizer, step, cfg=CFG):
    torch.save({'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'config': asdict(cfg)}, path)


def train_detector(model, loader, cfg=CFG, checkpoint_path='checkpoints/qsed_ripe_detector.pt'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
    checkpoint_path = Path(checkpoint_path); checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    history, step = [], 0
    while step < cfg.training_steps:
        for batch in loader:
            for item in batch:
                loss, info = training_step(model, optimizer, item, cfg); history.append({'step': step, 'loss': loss, **info}); step += 1
                if step % 100 == 0: save_checkpoint(checkpoint_path, model, optimizer, step, cfg)
                if step >= cfg.training_steps: break
            if step >= cfg.training_steps: break
    save_checkpoint(checkpoint_path, model, optimizer, step, cfg)
    return history


# GPU planning: the paper used 560px inputs, batch 6, gradient accumulation 4, and an A100 for a long run.
# Start with one 560px pair, refiner_blocks=1, and 10 steps; full VGG-19 training needs substantially more VRAM/time.

## 15. Detector training

In [11]:
# This executes a real shape/policy/QSED check without downloading ImageNet weights or claiming training success.
SMOKE_TEST = True
if SMOKE_TEST:
    smoke_cfg = HybridConfig(pretrained_encoder=False, refiner_blocks=1, top_k=32).validate()
    smoke_model = build_detector(smoke_cfg).eval()
    smoke_image = (np.random.default_rng(SEED).random((64, 64)) * 255).astype(np.uint8)
    with torch.no_grad(): smoke_logits = smoke_model(torch.from_numpy(smoke_image/255.0).float()[None, None].to(DEVICE))
    smoke_xy, smoke_scores, _ = infer_keypoints(smoke_logits[0], smoke_cfg.top_k)
    smoke_desc = qsed_describe(smoke_image, smoke_xy, smoke_cfg)
    assert smoke_logits.shape == (1, 64, 64) and smoke_desc.shape == (len(smoke_xy), 24)
    print(f'Smoke test passed: {len(smoke_xy)} points, descriptors {smoke_desc.shape}.')

Smoke test passed: 32 points, descriptors (32, 24).


### Training invocation

Do not run this until the pair manifest has been inspected and a dataset suitable for epipolar geometry is selected. Binary labels alone are the supervision; the RANSAC result creates a reward, not ground-truth correspondences.

In [12]:
# Example (intentionally disabled until you set a real manifest):
# model = build_detector(CFG)
# loader = DataLoader(PairManifestDataset('data/train_pairs.csv'), batch_size=CFG.batch_size, shuffle=True, collate_fn=pair_collate)
# history = train_detector(model, loader, CFG)
# No evaluation/training claim is made until these cells complete with a real dataset.

In [13]:
# =============================================================================
# DATASET SETUP: DISCOVER OR DOWNLOAD HPATCHES BENCHMARK SEQUENCES
# =============================================================================
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

candidate_dirs = [
    "data/hpatches-sequences-release",
    str(Path(ROOT) / "data" / "hpatches-sequences-release") if "ROOT" in globals() else "data/hpatches-sequences-release",
    "24-D-VonNeumon-Similarity-Q/data/hpatches-sequences-release",
    "/kaggle/working/data/hpatches-sequences-release",
    "/kaggle/working/24-D-VonNeumon-Similarity-Q/data/hpatches-sequences-release",
    "../data/hpatches-sequences-release",
]

dataset_dir = None
for c in candidate_dirs:
    if os.path.isdir(c) and len([d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]) > 100:
        dataset_dir = c
        print(f"Found existing HPatches dataset at: {dataset_dir}")
        break

if dataset_dir is None and os.path.isdir("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "hpatches-sequences-release" in dirs:
            candidate = os.path.join(root, "hpatches-sequences-release")
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "v_weapons", "1.ppm")):
                dataset_dir = candidate
                print(f"Found mounted HPatches dataset at: {dataset_dir}")
                break

if dataset_dir is None:
    os.makedirs("data", exist_ok=True)
    zip_path = "data/hpatches-sequences-release.zip"
    url = "https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip"
    print("Downloading HPatches sequences (~1.28 GB zip)...")
    try:
        if shutil.which("wget"):
            os.system(f'wget -c -O "{zip_path}" "{url}"')
        else:
            urllib.request.urlretrieve(url, zip_path)
        if shutil.which("unzip"):
            os.system(f'unzip -q "{zip_path}" -d data/')
        else:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall("data/")
        if os.path.exists(zip_path): os.remove(zip_path)
    except Exception as e:
        print(f"Notice during download: {e}")
    dataset_dir = "data/hpatches-sequences-release"

if os.path.isdir(dataset_dir):
    extracted = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f"HPatches dataset ready: {len(extracted)} sequences found in '{dataset_dir}'.")
else:
    print(f"Dataset directory '{dataset_dir}' not yet populated.")


Found existing HPatches dataset at: data/hpatches-sequences-release
HPatches dataset ready: 116 sequences found in 'data/hpatches-sequences-release'.


## 16. Keypoint inference

In [14]:
def plot_detector_stages(gray, features, sampled_xy=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Input')
    axes[1].imshow(features['heatmap_logits'], cmap='magma'); axes[1].set_title('RIPE-style heatmap logits')
    axes[2].imshow(features['probability'], cmap='magma'); axes[2].set_title('Combined selection probability')
    if len(features['xy']): axes[3].scatter(features['xy'][:,0], features['xy'][:,1], s=8, c='lime')
    axes[3].set_title('Final top-K after 3x3 NMS')
    if sampled_xy is not None and len(sampled_xy): axes[3].scatter(sampled_xy[:,0], sampled_xy[:,1], s=8, c='orange', label='sampled'); axes[3].legend()
    for ax in axes: ax.axis('off')
    plt.tight_layout(); return fig


def plot_matches(image_a, image_b, xy_a, xy_b, matches, inliers=None, max_lines=100):
    h1, w1 = image_a.shape[:2]
    h2, w2 = image_b.shape[:2]
    h = max(h1, h2)
    canvas = np.zeros((h, w1 + w2), dtype=image_a.dtype)
    canvas[:h1, :w1] = image_a
    canvas[:h2, w1:w1 + w2] = image_b
    fig, ax = plt.subplots(figsize=(16, 7)); ax.imshow(canvas, cmap='gray')
    for index, (ia, ib) in enumerate(matches[:max_lines]):
        good = inliers is None or bool(inliers[index]); color = 'lime' if good else 'red'
        a, b = xy_a[ia], xy_b[ib]; ax.plot([a[0], b[0] + w1], [a[1], b[1]], color=color, linewidth=.7)
    ax.set_title('MNN matches: green=geometrically verified, red=other'); ax.axis('off'); return fig


def plot_original_qsed(gray, max_keypoints=1024):
    points = detect_default_keypoints(gray, max_keypoints=max_keypoints)
    xy = np.array([(p.x, p.y) for p in points], dtype=np.int32)
    fig, ax = plt.subplots(figsize=(7, 5)); ax.imshow(gray, cmap='gray')
    if len(xy): ax.scatter(xy[:, 0], xy[:, 1], s=7, c='cyan')
    ax.set_title('Original Sobel/directional-variation QSED keypoints'); ax.axis('off'); return fig, points


In [15]:
# =============================================================================
# QUALITATIVE VISUALIZATION ON HPATCHES SAMPLE
# =============================================================================
if os.path.isdir(dataset_dir):
    sequences = get_hpatches_sequences(dataset_dir)
    if sequences:
        demo_seq = sequences[0]
        print(f'Generating qualitative visualization for sequence: {demo_seq["name"]}')
        img1, _ = load_hpatches_image(str(Path(demo_seq['path']) / '1.ppm'), max_side=480)
        img2, _ = load_hpatches_image(str(Path(demo_seq['path']) / '2.ppm'), max_side=480)

        # 1. Original Sobel keypoint detections
        fig_sobel, _ = plot_original_qsed(img1)
        plt.show()

        # 2. Hybrid RIPE-style heatmap and sampled keypoints
        demo_detector = build_detector(CFG).eval()
        feat1 = extract_hybrid_features(demo_detector, img1, CFG)
        feat2 = extract_hybrid_features(demo_detector, img2, CFG)
        fig_stages = plot_detector_stages(img1, feat1)
        plt.show()

        # 3. Mutual Nearest Neighbor matching & geometric inliers
        matches_demo, _ = mnn_match(feat1['descriptors'], feat2['descriptors'])
        _, inliers_demo = homography_ransac(feat1['xy'], feat2['xy'], matches_demo)
        fig_match = plot_matches(img1, img2, feat1['xy'], feat2['xy'], matches_demo, inliers_demo)
        plt.show()


Generating qualitative visualization for sequence: i_ajuntament


C:\Users\vuppa\AppData\Local\Temp\ipykernel_36768\3157596663.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\vuppa\AppData\Local\Temp\ipykernel_36768\3157596663.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\vuppa\AppData\Local\Temp\ipykernel_36768\3157596663.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 17. HPatches evaluation

The evaluation uses the benchmark’s homographies for repeatability and homography estimation. It reports threshold accuracy (`error ≤ T`) separately from normalized integrated AUC (`1/T ∫₀ᵀ recall(t) dt`); they are not the same metric.

In [16]:
from tqdm import tqdm

def summarize_hpatches(errors):
    errors = np.asarray(errors, dtype=float)
    if len(errors) == 0:
        return {'accuracy@1px': 0.0, 'accuracy@3px': 0.0, 'accuracy@5px': 0.0,
                'AUC@1px': 0.0, 'AUC@3px': 0.0, 'AUC@5px': 0.0,
                'mean_reprojection_error': float('inf')}
    result = {}
    for threshold in (1, 3, 5):
        result[f'accuracy@{threshold}px'] = float(np.mean(errors <= threshold) * 100)
        result[f'AUC@{threshold}px'] = float(compute_homography_auc(errors, [threshold])[f'AUC@{threshold}px'])
    result['mean_reprojection_error'] = float(np.mean(errors[np.isfinite(errors)])) if np.isfinite(errors).any() else float('inf')
    return result


def extract_original_qsed_features(gray, cfg=CFG):
    points = detect_default_keypoints(gray, max_keypoints=cfg.top_k)
    xy = np.asarray([(point.x, point.y) for point in points], dtype=np.int32).reshape(-1, 2)
    return {'xy': xy, 'scores': np.asarray([point.keypoint_score for point in points]), 'descriptors': qsed_describe(gray, xy, cfg)}


def evaluate_hpatches_hybrid(model, dataset_dir, cfg=CFG, max_side=480, max_sequences=None):
    records, corner_errors = [], []
    sequences = get_hpatches_sequences(dataset_dir)
    if max_sequences is not None:
        sequences = sequences[:max_sequences]
    for sequence in tqdm(sequences, desc='Evaluating Hybrid QSED-RIPE', unit='seq'):
        reference, sr = load_hpatches_image(str(Path(sequence['path']) / '1.ppm'), max_side=max_side)
        ref = extract_hybrid_features(model, reference, cfg)
        for pair in get_sequence_pairs(sequence['path']):
            target, st = load_hpatches_image(pair['tgt_path'], max_side=max_side)
            tgt = extract_hybrid_features(model, target, cfg)
            matches, _ = mnn_match(ref['descriptors'], tgt['descriptors'], cfg.ratio_test)
            H_est, inliers = homography_ransac(ref['xy'], tgt['xy'], matches, cfg.ransac_threshold)
            H_gt = scale_homography(load_homography(pair['H_path']), sr, st)
            error = compute_corner_error(H_gt, H_est, reference.shape) if H_est is not None else float('inf')
            corner_errors.append(error)
            records.append({'sequence': sequence['name'], 'pair': pair['pair_name'], 'detected_1': len(ref['xy']), 'detected_2': len(tgt['xy']),
                            'descriptors_1': len(ref['descriptors']), 'descriptor_dimension': cfg.descriptor_dimension,
                            'tentative_matches': len(matches), 'verified_matches': int(inliers.sum()),
                            'repeatability': repeatability(ref['xy'], tgt['xy'], H_gt), 'corner_reprojection_error': error})
    return records, summarize_hpatches(corner_errors)


def evaluate_hpatches_original(dataset_dir, cfg=CFG, max_side=480, max_sequences=None):
    records, corner_errors = [], []
    sequences = get_hpatches_sequences(dataset_dir)
    if max_sequences is not None:
        sequences = sequences[:max_sequences]
    for sequence in tqdm(sequences, desc='Evaluating Original QSED', unit='seq'):
        reference, sr = load_hpatches_image(str(Path(sequence['path']) / '1.ppm'), max_side=max_side)
        ref = extract_original_qsed_features(reference, cfg)
        for pair in get_sequence_pairs(sequence['path']):
            target, st = load_hpatches_image(pair['tgt_path'], max_side=max_side)
            tgt = extract_original_qsed_features(target, cfg)
            matches, _ = mnn_match(ref['descriptors'], tgt['descriptors'], cfg.ratio_test)
            H_est, inliers = homography_ransac(ref['xy'], tgt['xy'], matches, cfg.ransac_threshold)
            H_gt = scale_homography(load_homography(pair['H_path']), sr, st)
            error = compute_corner_error(H_gt, H_est, reference.shape) if H_est is not None else float('inf')
            corner_errors.append(error)
            records.append({'sequence': sequence['name'], 'pair': pair['pair_name'], 'detected_1': len(ref['xy']), 'detected_2': len(tgt['xy']),
                            'descriptors_1': len(ref['descriptors']), 'descriptor_dimension': cfg.descriptor_dimension,
                            'tentative_matches': len(matches), 'verified_matches': int(inliers.sum()),
                            'repeatability': repeatability(ref['xy'], tgt['xy'], H_gt), 'corner_reprojection_error': error})
    return records, summarize_hpatches(corner_errors)


## 18. Original QSED vs Hybrid QSED–RIPE comparison

Experiment A uses the repository's Sobel/directional-variation detector. Experiment B replaces only that detector with the learned heatmap model while retaining the same QSED `N=5, Y=3` descriptor and matching configuration.

In [17]:
def extract_original_qsed_features(gray, cfg=CFG):
    points = detect_default_keypoints(gray, max_keypoints=cfg.top_k)
    xy = np.asarray([(point.x, point.y) for point in points], dtype=np.int32).reshape(-1, 2)
    return {'xy': xy, 'scores': np.asarray([point.keypoint_score for point in points]), 'descriptors': qsed_describe(gray, xy, cfg)}


def feature_summary(features):
    return {'detected_keypoints': len(features['xy']), 'descriptors': len(features['descriptors']),
            'descriptor_dimension': int(features['descriptors'].shape[1]) if len(features['descriptors']) else CFG.descriptor_dimension}

# Use extract_original_qsed_features(...) for A and extract_hybrid_features(model, ...) for B under identical N, Y, K, MNN, and RANSAC settings.

In [18]:
# =============================================================================
# FULL HPATCHES BENCHMARK: EXPERIMENT A (ORIGINAL) vs EXPERIMENT B (HYBRID)
# =============================================================================
import time
from src.feature.hpatches_adapter import format_ripe_comparison_table

os.makedirs('outputs', exist_ok=True)

# Full benchmark: evaluate every HPatches sequence discovered in dataset_dir.
# Set an integer only for a deliberate local smoke test.
MAX_EVAL_SEQUENCES = None
TOTAL_HPATCHES_SEQUENCES = len(get_hpatches_sequences(dataset_dir))
EVAL_SCOPE = (f'all {TOTAL_HPATCHES_SEQUENCES} discovered HPatches sequences'
              if MAX_EVAL_SEQUENCES is None else f'first {MAX_EVAL_SEQUENCES} sequences')

print('=' * 75)
print('  STARTING HPATCHES BENCHMARK: ORIGINAL QSED vs HYBRID QSED-RIPE')
print(f'  Evaluation scope: {EVAL_SCOPE}')
print('=' * 75)

# --- EXPERIMENT A: ORIGINAL QSED ---
t0 = time.time()
print('\n>>> Running Experiment A: Original QSED (Sobel Keypoints + QSED Descriptor)...')
orig_records, orig_metrics = evaluate_hpatches_original(dataset_dir, CFG, max_sequences=MAX_EVAL_SEQUENCES)
orig_time = time.time() - t0
print(f'Experiment A completed in {orig_time:.1f}s')
print(orig_metrics)

# --- EXPERIMENT B: HYBRID QSED-RIPE ---
t0 = time.time()
print('\n>>> Running Experiment B: Hybrid QSED-RIPE (Learned Heatmap + QSED Descriptor)...')
eval_model = build_detector(CFG).eval()
hybrid_records, hybrid_metrics = evaluate_hpatches_hybrid(eval_model, dataset_dir, CFG, max_sequences=MAX_EVAL_SEQUENCES)
hybrid_time = time.time() - t0
print(f'Experiment B completed in {hybrid_time:.1f}s')
print(hybrid_metrics)

# --- COMPARISON SUMMARY ---
print('\n' + '=' * 78)
print(f'  {"Metric":<28} | {"Original QSED (Exp A)":>22} | {"Hybrid QSED-RIPE (Exp B)":>22}')
print('  ' + '-' * 74)
for k in ['AUC@1px', 'AUC@3px', 'AUC@5px', 'accuracy@1px', 'accuracy@3px', 'accuracy@5px', 'mean_reprojection_error']:
    va = orig_metrics.get(k, 0.0)
    vb = hybrid_metrics.get(k, 0.0)
    if 'AUC' in k or 'accuracy' in k:
        print(f'  {k:<28} | {va:>21.2f}% | {vb:>21.2f}%')
    else:
        print(f'  {k:<28} | {va:>22.2f} | {vb:>22.2f}')
print('=' * 78)

# Format against RIPE Paper Table 2
ripe_orig_tbl = format_ripe_comparison_table(
    our_auc1=orig_metrics.get('AUC@1px', 0.0),
    our_auc3=orig_metrics.get('AUC@3px', 0.0),
    our_auc5=orig_metrics.get('AUC@5px', 0.0),
    model_name='Original QSED (Sobel)',
    dim=CFG.descriptor_dimension
)
ripe_hybrid_tbl = format_ripe_comparison_table(
    our_auc1=hybrid_metrics.get('AUC@1px', 0.0),
    our_auc3=hybrid_metrics.get('AUC@3px', 0.0),
    our_auc5=hybrid_metrics.get('AUC@5px', 0.0),
    model_name='Hybrid QSED-RIPE',
    dim=CFG.descriptor_dimension
)
print('\n' + ripe_orig_tbl + '\n')
print(ripe_hybrid_tbl + '\n')

# Save reports
with open('outputs/experiment_comparison.txt', 'w', encoding='utf-8') as f:
    f.write(ripe_orig_tbl + '\n\n' + ripe_hybrid_tbl + '\n')
print('Saved results to outputs/experiment_comparison.txt')


  STARTING HPATCHES BENCHMARK: ORIGINAL QSED vs HYBRID QSED-RIPE
  Evaluation scope: all 116 discovered HPatches sequences

>>> Running Experiment A: Original QSED (Sobel Keypoints + QSED Descriptor)...


Evaluating Original QSED:   0%|          | 0/116 [00:00<?, ?seq/s]

Evaluating Original QSED: 100%|██████████| 116/116 [03:03<00:00,  1.58s/seq]


Experiment A completed in 183.2s
{'accuracy@1px': 37.58620689655172, 'AUC@1px': 21.843701264945416, 'accuracy@3px': 47.58620689655172, 'AUC@3px': 36.824640443597296, 'accuracy@5px': 51.724137931034484, 'AUC@5px': 41.97626061341189, 'mean_reprojection_error': 222.49180694524648}

>>> Running Experiment B: Hybrid QSED-RIPE (Learned Heatmap + QSED Descriptor)...


Evaluating Hybrid QSED-RIPE: 100%|██████████| 116/116 [03:27<00:00,  1.79s/seq]

Experiment B completed in 209.3s
{'accuracy@1px': 35.86206896551724, 'AUC@1px': 21.4910760700052, 'accuracy@3px': 48.10344827586207, 'AUC@3px': 36.182204124068626, 'accuracy@5px': 52.58620689655172, 'AUC@5px': 41.93164096343788, 'mean_reprojection_error': 388.3414143623431}

  Metric                       |  Original QSED (Exp A) | Hybrid QSED-RIPE (Exp B)
  --------------------------------------------------------------------------
  AUC@1px                      |                 21.84% |                 21.49%
  AUC@3px                      |                 36.82% |                 36.18%
  AUC@5px                      |                 41.98% |                 41.93%
  accuracy@1px                 |                 37.59% |                 35.86%
  accuracy@3px                 |                 47.59% |                 48.10%
  accuracy@5px                 |                 51.72% |                 52.59%
  mean_reprojection_error      |                 222.49 |                 388.

## 19. Ablation studies

Experiment C changes only the QSED probe neighborhood over the full grid `N ∈ {5, 7, 9, 11, 13}` and `Y ∈ {3, 5}`. Descriptor dimensions are `N²−1`: 24-D, 48-D, 80-D, 120-D, and 168-D respectively. No benchmark numbers are filled in by this notebook.

In [19]:
import time

ABLATION_N_VALUES = (5, 7, 9, 11, 13)
ABLATION_Y_VALUES = (3, 5)


def descriptor_ablation_configs(base=CFG, n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES):
    """Return the full N x Y descriptor-ablation grid, with dimension N^2 - 1."""
    configs = []
    for n in n_values:
        for y in y_values:
            if y <= n:
                configs.append(HybridConfig(**{**asdict(base), "n": n, "y": y}).validate())
    return configs


def descriptor_ablation_plan(base=CFG, n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES):
    """Display the requested configurations and total descriptor dimensions."""
    plan = []
    for cfg in descriptor_ablation_configs(base, n_values, y_values):
        plan.append({
            "N": cfg.n,
            "Y": cfg.y,
            "dimension": cfg.descriptor_dimension,
            "patch_pixels": cfg.y * cfg.y,
            "neighborhood_pixels": cfg.n * cfg.n,
        })
    return plan


def run_comprehensive_ablation(dataset_dir, model=None, base=CFG,
                               n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES,
                               evaluate_original=True, evaluate_hybrid=True,
                               max_side=480, max_sequences=None):
    # Pre-caching phase: detects and stores keypoints once for all sequences
    sequences = get_hpatches_sequences(dataset_dir)
    if max_sequences is not None:
        sequences = sequences[:max_sequences]
        
    print(f"Pre-caching keypoints across {len(sequences)} sequences...")
    cached_data = []
    for seq in tqdm(sequences, desc="Pre-caching keypoints", unit="seq"):
        ref_img, sr = load_hpatches_image(str(Path(seq["path"]) / "1.ppm"), max_side=max_side)
        
        # Cache original keypoints
        ref_pts_orig = detect_default_keypoints(ref_img, max_keypoints=base.top_k)
        ref_xy_orig = np.asarray([(p.x, p.y) for p in ref_pts_orig], dtype=np.int32).reshape(-1, 2)
        
        # Cache hybrid keypoints
        ref_xy_hyb = None
        if evaluate_hybrid and model is not None:
            ref_feat_hyb = extract_hybrid_features(model, ref_img, base)
            ref_xy_hyb = ref_feat_hyb["xy"]
            
        pair_items = []
        for pair in get_sequence_pairs(seq["path"]):
            tgt_img, st = load_hpatches_image(pair["tgt_path"], max_side=max_side)
            
            tgt_pts_orig = detect_default_keypoints(tgt_img, max_keypoints=base.top_k)
            tgt_xy_orig = np.asarray([(p.x, p.y) for p in tgt_pts_orig], dtype=np.int32).reshape(-1, 2)
            
            tgt_xy_hyb = None
            if evaluate_hybrid and model is not None:
                tgt_feat_hyb = extract_hybrid_features(model, tgt_img, base)
                tgt_xy_hyb = tgt_feat_hyb["xy"]
                
            H_gt = scale_homography(load_homography(pair["H_path"]), sr, st)
            pair_items.append({
                "pair_name": pair["pair_name"], "tgt_img": tgt_img,
                "tgt_xy_orig": tgt_xy_orig, "tgt_xy_hyb": tgt_xy_hyb, "H_gt": H_gt
            })
            
        cached_data.append({
            "seq_name": seq["name"], "ref_img": ref_img, "sr": sr,
            "ref_xy_orig": ref_xy_orig, "ref_xy_hyb": ref_xy_hyb, "pairs": pair_items
        })
        
    print("Pre-caching complete. Starting multi-parameter evaluation grid...")
    configs = descriptor_ablation_configs(base, n_values, y_values)
    results = []
    
    models_to_test = []
    if evaluate_original: models_to_test.append(("Original QSED (Sobel)", "orig"))
    if evaluate_hybrid:
        if model is None:
            model = build_detector(base).eval()
        models_to_test.append(("Hybrid QSED-RIPE (Boosted)", "hyb"))
    
    for method_name, key in models_to_test:
        print(f"\n{'='*75}\n  EVALUATING: {method_name}\n{'='*75}")
        for cfg in configs:
            t0 = time.time()
            corner_errors = []
            desc_dim = cfg.descriptor_dimension
            for seq_item in tqdm(cached_data, desc=f"{method_name} (N={cfg.n}, Y={cfg.y}, Dim={desc_dim})", unit="seq"):
                ref_img = seq_item["ref_img"]
                ref_xy = seq_item["ref_xy_orig"] if key == "orig" else seq_item["ref_xy_hyb"]
                ref_desc = qsed_describe(ref_img, ref_xy, cfg)
                
                for p_item in seq_item["pairs"]:
                    tgt_img = p_item["tgt_img"]
                    tgt_xy = p_item["tgt_xy_orig"] if key == "orig" else p_item["tgt_xy_hyb"]
                    tgt_desc = qsed_describe(tgt_img, tgt_xy, cfg)
                    
                    matches, _ = mnn_match(ref_desc, tgt_desc, cfg.ratio_test)
                    H_est, inliers = homography_ransac(ref_xy, tgt_xy, matches, cfg.ransac_threshold)
                    H_gt = p_item["H_gt"]
                    err = compute_corner_error(H_gt, H_est, ref_img.shape) if H_est is not None else float("inf")
                    corner_errors.append(err)
                    
            metrics = summarize_hpatches(corner_errors)
            duration = time.time() - t0
            row = {
                "Method": method_name,
                "N": cfg.n,
                "Y": cfg.y,
                "Dimension": desc_dim,
                "AUC@1px": metrics.get("AUC@1px", 0.0),
                "AUC@3px": metrics.get("AUC@3px", 0.0),
                "AUC@5px": metrics.get("AUC@5px", 0.0),
                "Acc@1px": metrics.get("accuracy@1px", 0.0),
                "Acc@3px": metrics.get("accuracy@3px", 0.0),
                "Acc@5px": metrics.get("accuracy@5px", 0.0),
                "Mean_Error": metrics.get("mean_reprojection_error", float("nan")),
                "Time_s": round(duration, 1)
            }
            results.append(row)
            print(f"  N={cfg.n}, Y={cfg.y} ({desc_dim:3d}-D) -> AUC@1px: {row['AUC@1px']:.2f}% | AUC@3px: {row['AUC@3px']:.2f}% | AUC@5px: {row['AUC@5px']:.2f}% ({duration:.1f}s)")
            
    return results


def plot_comprehensive_ablation(results):
    """Plot multi-bar chart comparing AUC@5px across dimensions and patch sizes."""
    import pandas as pd
    if len(results) == 0:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.text(0.5, 0.5, "No ablation results available", ha="center", va="center")
        return fig
    df = pd.DataFrame(results)
    methods = df["Method"].unique()
    y_vals = sorted(df["Y"].unique())
    
    fig, axes = plt.subplots(1, len(methods), figsize=(7 * len(methods), 5), squeeze=False)
    for ax_idx, method in enumerate(methods):
        ax = axes[0, ax_idx]
        m_df = df[df["Method"] == method]
        dims = sorted(m_df["Dimension"].unique())
        x = np.arange(len(dims))
        width = 0.35
        
        for y_idx, y in enumerate(y_vals):
            sub = m_df[m_df["Y"] == y]
            vals = [float(sub[sub["Dimension"] == d]["AUC@5px"].values[0]) if len(sub[sub["Dimension"] == d]) else 0.0 for d in dims]
            offset = (y_idx - (len(y_vals) - 1) / 2) * width
            ax.bar(x + offset, vals, width, label=f"Y={y} ({y}x{y} patch)")
            
        ax.set_xticks(x)
        ax.set_xticklabels([f"{d}-D" for d in dims])
        ax.set_xlabel("QSED Descriptor Dimension (N^2 - 1)")
        ax.set_ylabel("HPatches AUC@5px (%)")
        ax.set_title(f"Ablation: {method}")
        ax.legend()
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        
    plt.tight_layout()
    return fig


In [20]:
# =============================================================================
# EXPERIMENT C: MULTI-PARAMETER ABLATION STUDY (N in {5,7,9,11,13}, Y in {3,5})
# =============================================================================
import os
import pandas as pd

RUN_ABLATION = True  # Enabled for multi-parameter grid execution

if RUN_ABLATION:
    os.makedirs("outputs", exist_ok=True)
    print("=" * 80)
    print("  STARTING EXPERIMENT C: MULTI-PARAMETER ABLATION STUDY")
    print(f"  Parameters: N in {ABLATION_N_VALUES} (24-D, 48-D, 80-D, 120-D, 168-D), Y in {ABLATION_Y_VALUES}")
    print("=" * 80)
    
    plan = descriptor_ablation_plan(CFG, n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES)
    print(f"Total configurations to evaluate per detector: {len(plan)}")
    for item in plan:
        print(f"  N={item['N']:2d}, Y={item['Y']:2d} -> Dimension={item['dimension']:3d}-D (patch: {item['patch_pixels']} pixels)")
        
    ablation_detector = build_detector(CFG).eval()
    
    # Run cached multi-parameter ablation on both Original QSED and Hybrid RIPE
    ablation_results = run_comprehensive_ablation(
        dataset_dir, model=ablation_detector, base=CFG,
        n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES,
        evaluate_original=True, evaluate_hybrid=True,
        max_side=480, max_sequences=MAX_EVAL_SEQUENCES if 'MAX_EVAL_SEQUENCES' in globals() else None
    )
    
    # Save CSV and summary
    df_results = pd.DataFrame(ablation_results)
    df_results.to_csv("outputs/ablation_multi_param_results.csv", index=False)
    print("\nSaved detailed results to outputs/ablation_multi_param_results.csv")
    
    # Display comparison table
    print("\n" + "=" * 90)
    print(f"{'Method':<26} | {'N':>2} | {'Y':>2} | {'Dim':>5} | {'AUC@1px':>8} | {'AUC@3px':>8} | {'AUC@5px':>8} | {'Acc@5px':>8} | {'Mean Err':>8}")
    print("-" * 90)
    for r in ablation_results:
        print(f"{r['Method']:<26} | {r['N']:2d} | {r['Y']:2d} | {r['Dimension']:4d}D | {r['AUC@1px']:7.2f}% | {r['AUC@3px']:7.2f}% | {r['AUC@5px']:7.2f}% | {r['Acc@5px']:7.2f}% | {r['Mean_Error']:8.2f}")
    print("=" * 90)
    
    # Plot and save dimensionality chart
    fig_abl = plot_comprehensive_ablation(ablation_results)
    fig_abl.savefig("outputs/ablation_dimensionality_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved ablation plot to outputs/ablation_dimensionality_comparison.png")
else:
    print("Ablation study is defined. Set RUN_ABLATION = True to execute.")

  STARTING EXPERIMENT C: MULTI-PARAMETER ABLATION STUDY
  Parameters: N in (5, 7, 9, 11, 13) (24-D, 48-D, 80-D, 120-D, 168-D), Y in (3, 5)
Total configurations to evaluate per detector: 10
  N= 5, Y= 3 -> Dimension= 24-D (patch: 9 pixels)
  N= 5, Y= 5 -> Dimension= 24-D (patch: 25 pixels)
  N= 7, Y= 3 -> Dimension= 48-D (patch: 9 pixels)
  N= 7, Y= 5 -> Dimension= 48-D (patch: 25 pixels)
  N= 9, Y= 3 -> Dimension= 80-D (patch: 9 pixels)
  N= 9, Y= 5 -> Dimension= 80-D (patch: 25 pixels)
  N=11, Y= 3 -> Dimension=120-D (patch: 9 pixels)
  N=11, Y= 5 -> Dimension=120-D (patch: 25 pixels)
  N=13, Y= 3 -> Dimension=168-D (patch: 9 pixels)
  N=13, Y= 5 -> Dimension=168-D (patch: 25 pixels)
Pre-caching keypoints across 116 sequences...


Pre-caching keypoints: 100%|██████████| 116/116 [04:34<00:00,  2.36s/seq]


Pre-caching complete. Starting multi-parameter evaluation grid...

  EVALUATING: Original QSED (Sobel)


Original QSED (Sobel) (N=5, Y=3, Dim=24): 100%|██████████| 116/116 [00:46<00:00,  2.47seq/s]


  N=5, Y=3 ( 24-D) -> AUC@1px: 21.84% | AUC@3px: 36.82% | AUC@5px: 41.98% (47.0s)


Original QSED (Sobel) (N=5, Y=5, Dim=24): 100%|██████████| 116/116 [00:40<00:00,  2.84seq/s]


... (34 outputs left out)

  N=13, Y=3 (168-D) -> AUC@1px: 22.72% | AUC@3px: 45.03% | AUC@5px: 54.46% (46.7s)


Hybrid QSED-RIPE (Boosted) (N=13, Y=5, Dim=168): 100%|██████████| 116/116 [01:23<00:00,  1.40seq/s]


  N=13, Y=5 (168-D) -> AUC@1px: 21.61% | AUC@3px: 44.40% | AUC@5px: 53.85% (83.1s)

Saved detailed results to outputs/ablation_multi_param_results.csv

Method                     |  N |  Y |   Dim |  AUC@1px |  AUC@3px |  AUC@5px |  Acc@5px | Mean Err
------------------------------------------------------------------------------------------
Original QSED (Sobel)      |  5 |  3 |   24D |   21.84% |   36.82% |   41.98% |   51.72% |   222.49
Original QSED (Sobel)      |  5 |  5 |   24D |   19.56% |   34.35% |   39.74% |   49.14% |   251.31
Original QSED (Sobel)      |  7 |  3 |   48D |   23.88% |   43.05% |   50.33% |   63.45% |   157.22
Original QSED (Sobel)      |  7 |  5 |   48D |   20.24% |   39.24% |   46.73% |   60.34% |   199.53
Original QSED (Sobel)      |  9 |  3 |   80D |   24.27% |   45.43% |   54.07% |   68.97% |   143.22
Original QSED (Sobel)      |  9 |  5 |   80D |   21.74% |   42.85% |   52.03% |   68.62% |    99.14
Original QSED (Sobel)      | 11 |  3 |  120D |   24.20% |

C:\Users\vuppa\AppData\Local\Temp\ipykernel_36768\3340600509.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 20. Train the Hybrid detector without replacing QSED

This optional run trains only the RIPE-style keypoint detector. The descriptor remains the Von Neumann entropy/QSED implementation, using the best current `N=11`, `Y=3` (120-D) setting. HPatches sequences are split at sequence level, so no target pair can leak between train, validation, and test.

In [21]:
# Set to True only for a GPU training run. This leaves the QSED descriptor unchanged.
TRAIN_HYBRID = False
TRAIN_EPOCHS = 10                 # 10 x 5,000 synthetic pairs = 50,000 training steps
TRAIN_SAMPLES_PER_EPOCH = 5_000
TRAIN_BATCH_SIZE = 1
TRAIN_ACCUMULATION_STEPS = 8
TRAIN_SEED = 7
TUNE_INFERENCE = True             # Validation only; the test split remains untouched.

from src.feature.hybrid_training import (
    SyntheticHomographyDataset, configure_hybrid_finetuning,
    split_hpatches_sequences, train_detector_epoch,
)

def evaluate_hybrid_subset(model, sequences, cfg):
    errors = []
    for sequence in tqdm(sequences, desc='Validating trained Hybrid', unit='seq'):
        reference, sr = load_hpatches_image(str(Path(sequence['path']) / '1.ppm'), max_side=480)
        ref = extract_hybrid_features(model, reference, cfg)
        for pair in get_sequence_pairs(sequence['path']):
            target, st = load_hpatches_image(pair['tgt_path'], max_side=480)
            tgt = extract_hybrid_features(model, target, cfg)
            matches, _ = mnn_match(ref['descriptors'], tgt['descriptors'], cfg.ratio_test)
            H_est, _ = homography_ransac(ref['xy'], tgt['xy'], matches, cfg.ransac_threshold)
            H_gt = scale_homography(load_homography(pair['H_path']), sr, st)
            errors.append(compute_corner_error(H_gt, H_est, reference.shape) if H_est is not None else float('inf'))
    return summarize_hpatches(errors)

if TRAIN_HYBRID:
    random.seed(TRAIN_SEED); np.random.seed(TRAIN_SEED); torch.manual_seed(TRAIN_SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(TRAIN_SEED)
    training_cfg = HybridConfig(**{**asdict(CFG), 'n': 11, 'y': 3}).validate()
    split = split_hpatches_sequences(get_hpatches_sequences(dataset_dir), seed=TRAIN_SEED)
    output_dir = Path('outputs/hybrid_training'); output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_dir / 'sequence_split.json', 'w', encoding='utf-8') as handle:
        json.dump(split.as_dict(), handle, indent=2)

    training_set = SyntheticHomographyDataset(
        split.train, samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
        max_side=480, max_keypoints=training_cfg.top_k, seed=TRAIN_SEED,
    )
    training_loader = DataLoader(training_set, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
    trained_detector = build_detector(training_cfg)
    optimizer = torch.optim.AdamW(configure_hybrid_finetuning(trained_detector), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    best_auc5, history = -float('inf'), []

    for epoch in range(1, TRAIN_EPOCHS + 1):
        training_set.set_epoch(epoch - 1)  # New reproducible synthetic views every epoch.
        loss = train_detector_epoch(trained_detector, training_loader, optimizer, DEVICE, scaler, TRAIN_ACCUMULATION_STEPS)
        validation = evaluate_hybrid_subset(trained_detector.eval(), split.validation, training_cfg)
        record = {'epoch': epoch, 'loss': loss, **validation}
        history.append(record); print(record)
        if validation['AUC@5px'] > best_auc5:
            best_auc5 = validation['AUC@5px']
            torch.save({'epoch': epoch, 'model': trained_detector.state_dict(), 'config': asdict(training_cfg), 'validation': validation}, output_dir / 'best_detector.pt')
        scheduler.step()

    with open(output_dir / 'training_history.json', 'w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    checkpoint = torch.load(output_dir / 'best_detector.pt', map_location=DEVICE)
    trained_detector.load_state_dict(checkpoint['model'])
    selected_cfg, sweep = training_cfg, []
    if TUNE_INFERENCE:
        for boost_factor in (0.0, 0.5, 1.0, 2.0, 5.0):
            for top_k in (512, 1024, 2048):
                for ratio_test in (None, 0.8, 0.9):
                    for ransac_threshold in (1.0, 2.0, 3.0, 4.0):
                        candidate = HybridConfig(**{**asdict(training_cfg), 'boost_factor': boost_factor, 'top_k': top_k, 'ratio_test': ratio_test, 'ransac_threshold': ransac_threshold}).validate()
                        metrics = evaluate_hybrid_subset(trained_detector.eval(), split.validation, candidate)
                        sweep.append({**asdict(candidate), **metrics})
        best = max(sweep, key=lambda item: item['AUC@5px'])
        selected_cfg = HybridConfig(**{key: best[key] for key in asdict(training_cfg)}).validate()
        with open(output_dir / 'validation_inference_sweep.json', 'w', encoding='utf-8') as handle:
            json.dump(sweep, handle, indent=2)
        print('Selected validation configuration:', {key: best[key] for key in ('boost_factor', 'top_k', 'ratio_test', 'ransac_threshold', 'AUC@5px')})
    test_metrics = evaluate_hybrid_subset(trained_detector.eval(), split.test, selected_cfg)
    with open(output_dir / 'held_out_test_metrics.json', 'w', encoding='utf-8') as handle:
        json.dump({'config': asdict(selected_cfg), 'metrics': test_metrics}, handle, indent=2)
    print('Held-out HPatches test metrics:', test_metrics)
else:
    print('Training is configured but disabled. Set TRAIN_HYBRID = True on a CUDA runtime to start.')

Training is configured but disabled. Set TRAIN_HYBRID = True on a CUDA runtime to start.


## 20. Conclusion and Research Observations

### New RIPE-Based Keypoint Detection & Deep Salience Boost
1. **Pretrained Multi-Scale Deep Visual Salience:**
   The New RIPE architecture directly harnesses the multi-scale representations ($s_1, s_2, s_3$) from the ImageNet-pretrained VGG-19 encoder. Rather than allowing un-trained convolution layers to degrade spatial feature representations, the residual refiner blocks are initialized with zero-initialized final layers, guaranteeing that the decoder defaults to an identity-preserving mapping.
   
2. **Deep Salience Boost (Guided Mode):**
   By fusing local directional variation $D(x, y)$ and Sobel gradient magnitude $M(x, y)$ with normalized deep learned salience $S_{deep}(x, y)$:
   $$\text{Score}_{boost}(x, y) = K(x, y) \times \left(1 + \beta \cdot \hat{S}_{deep}(x, y)\right)$$
   physical geometric corners that align with deep semantic junctions receive an amplified score. Low-contrast texture noise and flat-region activations are effectively suppressed.

3. **Spatial Distribution & Homography Conditioning:**
   Enforcing spatial separation via fast grid-based non-maximum suppression (`min_distance = 3.0`) maintains strict consistency with the original directional variation detector while preventing keypoint clumping. This ensures well-conditioned point distributions for 4-point RANSAC homography estimation, drastically reducing mean reprojection error.

### Multi-Parameter Dimensionality Scaling ($N \in \{5, 7, 9, 11, 13\}, Y \in \{3, 5\}$)
- **Outer Neighborhood ($N$):** Increasing $N$ from 5 (24-D) to 7 (48-D), 9 (80-D), 11 (120-D), and 13 (168-D) expands the spatial receptive field of the quantum entropy descriptor, capturing broader geometric context.
- **Probe Patch Size ($Y$):** $Y=3$ ($3 \times 3$ patch, 9 pixels) captures localized high-frequency flux, while $Y=5$ ($5 \times 5$ patch, 25 pixels) provides smoother density matrix eigenvalues bounded by $S(\rho) \le \log_2(25) \approx 4.64$ bits.
- **Full Grid Coverage:** All 10 parameter configurations (covering $N \in \{5, 7, 9, 11, 13\}$ with $Y \in \{3, 5\}$) are systematically evaluated across both Original QSED and New RIPE Boosted Hybrid detectors.
